# Telugu Tokenizer Fertility & Fairness Audit
Run all cells top to bottom. Add API keys in **Cell 2** before running.

In [ ]:
# ── Cell 1: Clone repo & install dependencies ──────────────────────────────
!git clone https://github.com/vishnup22/telugu-tokenizer-audit.git
%cd telugu-tokenizer-audit
!pip install -q -r requirements.txt
!pip install -q sentencepiece  # for dravidian-gpt2-telugu

In [ ]:
# ── Cell 2: API keys (paste your keys here) ────────────────────────────────
import os

# Paste your keys between the quotes:
os.environ['ANTHROPIC_API_KEY'] = ''   # required for claude tokenizer
os.environ['OPENAI_API_KEY']    = ''   # required for openai-gpt4o tokenizer
os.environ['HF_TOKEN']          = ''   # optional, only needed for gated models

# Alternative: use Colab Secrets (Colab > Tools > Secrets)
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')

print('Keys set:', [k for k in ['ANTHROPIC_API_KEY','OPENAI_API_KEY','HF_TOKEN'] if os.environ.get(k)])

In [ ]:
# ── Cell 3: Collect corpus data ────────────────────────────────────────────
# Downloads ~1000 sentences each from Wikipedia, social media, then ITRANS-transliterates
!python scripts/collect_data.py --n-samples 1000

In [ ]:
# ── Cell 4: Create experiment directory ────────────────────────────────────
import datetime, os
exp_tag = datetime.datetime.now().strftime('%Y-%m-%d_colab')
exp_dir = f'experiments/{exp_tag}'
os.makedirs(f'{exp_dir}', exist_ok=True)
import shutil
shutil.copy('configs/default.yaml', f'{exp_dir}/config_snapshot.yaml')
print('Experiment dir:', exp_dir)

In [ ]:
# ── Cell 5: Run fertility audit (all 6 tokenizers) ─────────────────────────
# Claude calls will hit the API (~10 min first run; cached on re-runs)
# All other tokenizers are local BPE — fast
!python scripts/02_run_tokenizer_audit.py --config configs/default.yaml --experiment-dir {exp_dir}

In [ ]:
# ── Cell 6: Minimal pair audit ─────────────────────────────────────────────
!python scripts/03_run_minimal_pair_audit.py --config configs/default.yaml --experiment-dir {exp_dir}

In [ ]:
# ── Cell 7: Significance tests ─────────────────────────────────────────────
!python scripts/06_run_significance_tests.py --experiment-dir {exp_dir} --config configs/default.yaml

In [ ]:
# ── Cell 8: Generate figures & tables ──────────────────────────────────────
!python scripts/05_make_figures_and_tables.py --experiment-dir {exp_dir}

In [ ]:
# ── Cell 9: Print key results ──────────────────────────────────────────────
import pandas as pd

print('=== FERTILITY BY REGISTER ===')
df = pd.read_csv(f'{exp_dir}/results/fertility_by_register.csv')
pivot = df.pivot_table(index='tokenizer', columns='register', values='fertility_tokens_per_word').round(2)
pivot['script_gap'] = (pivot['native_informal'] / pivot['romanized_informal']).round(2)
print(pivot.sort_values('native_informal').to_string())

print('\n=== SIGNIFICANCE TESTS ===')
import json
sig = json.load(open(f'{exp_dir}/results/script_gap_significance.json'))
for tok, s in sig.items():
    print(f"{tok:15s} p={s['p_value']:.4f}  median_gap={s['median_gap']:.3f}  CI={s['ci_95']}")

print('\n=== MINIMAL PAIR BREAKDOWN ===')
mp = pd.read_csv(f'{exp_dir}/results/minimal_pair_fertility.csv')
print(mp.pivot_table(index='morph_type', columns='tokenizer', values='n_tokens', aggfunc='mean').round(2).to_string())

In [ ]:
# ── Cell 10: Download results as zip ───────────────────────────────────────
import shutil
from google.colab import files

zip_path = f'/tmp/telugu_audit_{exp_tag}'
shutil.make_archive(zip_path, 'zip', exp_dir)
files.download(zip_path + '.zip')
print('Downloaded!')